# Ein Baum, ein Ort, ein Zeitpunkt

Die Kette des Vortrags, einmal durchgerechnet: Vorrat, zwei Antriebskanäle,
Faltung zur Temperatur. Erst deterministisch, dann als Störungs-Ensemble.

Die Albedo-Werte stammen noch aus der Breitengrad-Ersatztabelle; die Raster
kommen mit Plan 2.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from nbs.constants import YEARS
from nbs.forcing import fallback_delta_alpha, fallback_q_sw, light_bulb_watts
from nbs.metrics import prob_warming, risk_premium, t_star
from nbs.pipeline import run_single_tree, run_single_tree_ensemble
from nbs.plots import plot_decomposition, plot_disturbance_fan, plot_tree_inputs
from nbs.species import get_species

FIG = '../docs/slides/figures'

## Die Glühbirne von Folie 14

Bevor irgendetwas gerechnet wird: stimmt die Zahl, die auf der Folie steht?

In [2]:
for key, lat in [('picea_abies', 60.0), ('tropical_mixed', 0.0)]:
    sp = get_species(key)
    w = light_bulb_watts(fallback_delta_alpha(lat), fallback_q_sw(lat), sp)
    print(f'{sp.name_de:32s} {lat:4.0f} Grad N: {w:5.1f} W je Baum')

Gemeine Fichte                     60 Grad N:  71.5 W je Baum
Tropischer Mischbestand             0 Grad N:  34.7 W je Baum


## Der boreale Fall

In [3]:
boreal = run_single_tree('picea_abies', lat_deg=60.0, scenario='ssp245')
print('t* =', t_star(boreal.delta_t, YEARS))
print('Delta T 2150 =', f'{boreal.delta_t[-1]:.3e} K je Baum')

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

t* = 2049.0
Delta T 2150 = -9.004e-14 K je Baum


In [4]:
plot_tree_inputs(boreal, f'{FIG}/e2_eingangsgroessen_boreal.png')
plot_decomposition(boreal, f'{FIG}/e3_zerlegung_boreal.png')

'../docs/slides/figures/e3_zerlegung_boreal.png'

## Der tropische Fall — dieselbe Rechnung, anderes Vorzeichen

In [5]:
tropisch = run_single_tree('tropical_mixed', lat_deg=0.0, scenario='ssp245')
plot_decomposition(tropisch, f'{FIG}/e3_zerlegung_tropisch.png')
print('t* tropisch =', t_star(tropisch.delta_t, YEARS))

t* tropisch = 2025.0


## Alle fünf Arten an ihren Standorten

Vorlage für Folie E5.

In [6]:
SITES = {
    'picea_abies': 60.0,
    'pinus_sylvestris': 52.0,
    'fagus_sylvatica': 50.0,
    'eucalyptus_grandis': 25.0,
    'tropical_mixed': 0.0,
}
results = {k: run_single_tree(k, lat) for k, lat in SITES.items()}

fig, ax = plt.subplots(figsize=(9, 5))
for key, r in results.items():
    ax.plot(r.years, r.delta_t * 1e15, label=f'{get_species(key).name_de} ({SITES[key]:.0f} Grad N)')
ax.axhline(0, color='0.6', lw=0.8)
ax.set_xlabel('Jahr'); ax.set_ylabel('Delta T [1e-15 K je Baum]')
ax.legend(frameon=False); ax.grid(alpha=0.25)
fig.savefig(f'{FIG}/e5_arten.png', dpi=300, bbox_inches='tight')

## Störung als Risiko

1000 Läufe. Der Abstand zwischen Ensemblemittel und deterministischem Lauf
ist die Risikoprämie: um so viel schmeichelt sich eine Rechnung ohne
Störungsereignisse selbst.

In [7]:
det, ens = run_single_tree_ensemble('picea_abies', lat_deg=60.0, n_members=1000, seed=0)
print('P(Delta T > 0 in 2100) =', f"{prob_warming(ens, YEARS, 2100):.1%}")
print('Risikoprämie 2100      =', f"{risk_premium(ens, det.delta_t, YEARS, 2100):.3e} K")
plot_disturbance_fan(det, ens, f'{FIG}/e4_stoerungsfaecher_boreal.png')

P(Delta T > 0 in 2100) = 0.0%
Risikoprämie 2100      = 1.044e-15 K


'../docs/slides/figures/e4_stoerungsfaecher_boreal.png'

## Offen

- Albedo und Einstrahlung aus echten Rastern statt aus der Breitentabelle (Plan 2)
- Hochskalierung auf Flächenpotenziale und die Bastin-Korrekturkaskade (Plan 2)
- Übernahme der Abbildungen in die Präsentation (Plan 3)